# 데이터 수집

이건 이미 완료해서, 노션 속 데이터 쓰면 됩니당

# 데이터 전처리

In [ ]:
import re
from konlpy.tag import Okt
import pandas as pd

okt = Okt()

def clean_text(text):
    text = re.sub(r'[^가-힣\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# 부정어 도출 안하는거
def extract_keywords(text):
    if not text or not isinstance(text, str):
        return []

    text = re.sub(r'[^\w\s가-힣]', '', text)
    morphs = okt.pos(text, norm=True, stem=True)
    result = []

    for word, tag in morphs:
        if tag in ['Noun', 'Verb', 'Adjective']:
            result.append(word)

    return result

In [ ]:
# 부정어 도출할거면 이거 쓰면 됩니당
def extract_keywords_with_negation(text):
    if not text or not isinstance(text, str):
        return []

    text = re.sub(r'[^\w\s가-힣]', '', text)
    morphs = okt.pos(text, norm=True, stem=True)
    result = []
    i = 0

    while i < len(morphs):
        word, tag = morphs[i]

        # Case 1: '지 않다' or '지 못하다'
        if word == '지' and i + 1 < len(morphs):
            next_word, next_tag = morphs[i + 1]
            if next_word in ['않다', '못하다'] and next_tag == 'Verb':
                if result:
                    prev = result.pop()
                    result.append(f'NOT_{prev}')
                i += 2
                continue

        # Case 2: '않다', '못하다', '없다' (보조용언/형용사 → 앞 단어 부정)
        if word in ['않다', '못하다', '없다'] and tag in ['Verb', 'Adjective']:
            if result:
                prev = result.pop()
                result.append(f'NOT_{prev}')
            i += 1
            continue

        # ✅ Case 3: 부정 부사 (안, 못, 아니)
        if word in ['안', '못', '아니'] and tag in ['Adverb', 'Noun']:
            if i + 1 < len(morphs):
                next_word, next_tag = morphs[i + 1]

                if next_tag in ['Verb', 'Adjective']:
                    # 👇 수정된 부분: '나다/보이다/들리다'는 앞 명사 부정
                    if result and result[-1] and next_word in ['나다', '보이다', '들리다']:
                        prev = result[-1]  # pop하지 않고 유지
                        result.append(f'NOT_{next_word}')  # 동사에 부정
                    else:
                        result.append(f'NOT_{next_word}')
                    i += 2
                    continue


        # 일반 키워드: 명사/동사/형용사
        if tag in ['Noun', 'Verb', 'Adjective']:
            result.append(word)

        i += 1

    return result

In [ ]:
# 폴더 경로만 주의해서 지정하면 됩니당
import os
import pandas as pd

file_names = os.listdir('./data/전처리_완료')
for i in file_names:
    print(i)
    df = pd.read_csv('./data/전처리_완료/' + i)
    df['token'] = df['리뷰'].apply(clean_text).apply(extract_keywords)
    # df['token'] = df['리뷰'].apply(clean_text).apply(extract_keywords_with_negation)
    df.to_csv('./data/전처리_완료/구분X_' + i, index=False)

In [ ]:
df_ddgb_1 = pd.read_csv('./data/전처리_완료/구분X_남도예담떡갈비_리뷰_250607.csv')
df_ddgb_2 = pd.read_csv('./data/전처리_완료/구분X_돌쇠네떡갈비_리뷰_250624.csv')
df_ddgb_3 = pd.read_csv('./data/전처리_완료/구분X_와룡총각떡갈비_리뷰_250624.csv')

In [ ]:
# 중복 토큰 제거
def remove_duplicated_token(tokens):
    seen = set()
    result = []
    for token in tokens:
        if token not in seen:
            seen.add(token)
            result.append(token)
    return result


# 파일로 저장후 다시 불러왔을 때, 리스트 형태가 아닌 str로 표시되는 문제 해결
import ast
def clean_dataframe(df):
    df['token'] = df['token'].apply(ast.literal_eval)
    df['token'] = df['token'].apply(remove_duplicated_token)
    return df.drop_duplicates(subset='리뷰', keep='first')

In [ ]:
df_ddgb_1 = clean_dataframe(df_ddgb_1)
df_ddgb_2 = clean_dataframe(df_ddgb_2)
df_ddgb_3 = clean_dataframe(df_ddgb_3)

# 빈도수 구하기

In [ ]:
from itertools import chain

all_tokens = list(chain.from_iterable(
    df['token'] for df in [df_ddgb_1, df_ddgb_2, df_ddgb_3]  # 또는 'tokens_dedup'
))

In [ ]:
# 빈도수 계산
from collections import Counter
token_freq = Counter(chain.from_iterable(all_tokens))

# DataFrame으로 보기 좋게 변환
freq_df = pd.DataFrame(token_freq.items(), columns=['토큰', '빈도수']).sort_values(by='빈도수', ascending=False)
freq_df

In [ ]:
!pip install openpyxl

In [ ]:
# 엑셀로 보는게 편해서.. 이거 보면서 유의미한 맛 표현 키워드 도출
#freq_df.to_excel('./data/구분X_토큰_빈도수.xlsx', index=False)

In [ ]:
from collections import Counter
from itertools import chain
import numpy as np
import pandas as pd

# 1. 맛 관련 키워드 리스트 (개별 단어)
taste_keywords = [
    '육즙','적다','부드럽다','작다','숯불','냄새','불향','깔끔하다',
    '적당하다','달다','기름','달','촉촉하다','짜다','가득하다','불맛','불',
    '간도','양념','크기','두께','식감','씹히다','담백하다','간이',
    '고추','느끼하다','강하다','짜지다','두툼'

]

# 2. 맛 그룹 딕셔너리 (필요한 단어만 그룹화, 예: '맑다'에 '맑다', '맑은' 묶기)
taste_groups = {
    '짠맛': ['짜다', '짜지다'],
    '달다': ['달다', '달'],
    '간': ['간도','간이']
}


# 1. 그룹화 키워드 추출
group_keywords = set()
for v in taste_groups.values():
    group_keywords.update(v)
group_names = list(taste_groups.keys())

# 2. 그룹 미포함 키워드만 추출
individual_keywords = [kw for kw in taste_keywords if kw not in group_keywords]

# for word in group_names + individual_keywords:
#     not_words.append('NOT_'+word)

In [ ]:
total_keywords = individual_keywords + group_names

# 3. 컬럼명 만들기
keyword_columns = [f"{kw}_비율" for kw in total_keywords]
base_columns = ['가게명', '리뷰수','키워드포함_리뷰수',  '평균평점', '맛있다_수', '맛있다_비율']
total_info_df = pd.DataFrame(columns=base_columns + keyword_columns)

# 4. 계산 루프
for temp in [df_ddgb_1, df_ddgb_2, df_ddgb_3]:
    review_count = len(temp)        
    temp = temp[temp['token'].apply(lambda tokens: any(token in tokens for token in total_keywords))]
    token_freq = Counter(chain.from_iterable(temp['token']))
    df_freq = pd.DataFrame(token_freq.items(), columns=['토큰', '빈도수'])


    review_count_keyword = len(temp)
    try:
        option = temp['세부옵션'].value_counts().idxmax()
    except:
        option = np.nan

    mean_score = temp['별점'].mean()
    delicious_count = df_freq[df_freq['토큰'] == '맛있다'].values[0][1] if '맛있다' in df_freq['토큰'].values else 0
    delicious_ratio = delicious_count / review_count_keyword if review_count_keyword > 0 else 0

    # 4-1. 그룹화 안 된 키워드 비율
    keyword_ratios = []
    for kw in individual_keywords:
        kw_count = temp['token'].apply(lambda tokens: kw in tokens).sum()
        kw_ratio = kw_count / review_count_keyword if review_count_keyword > 0 else 0
        keyword_ratios.append(kw_ratio)

    # 4-2. 그룹 비율
    for g in group_names:
        group_kw_list = taste_groups[g]
        group_count = temp['token'].apply(lambda tokens: any(kw in tokens for kw in group_kw_list)).sum()
        group_ratio = group_count / review_count_keyword if review_count_keyword > 0 else 0
        keyword_ratios.append(group_ratio)


    total_info_df.loc[len(total_info_df)] = [np.nan, review_count, review_count_keyword, mean_score,
                                             delicious_count, delicious_ratio] + keyword_ratios



# 가게명 기입
total_info_df['가게명'] = ['남도예담떡갈비', '돌쇠네떡갈비', '와룡총각떡갈비']
total_info_df = total_info_df.T
total_info_df.columns = total_info_df.iloc[0]
total_info_df = total_info_df[1:]  # 첫 번째 행 제거
total_info_df = total_info_df.loc[~(total_info_df == 0).all(axis=1)]

In [ ]:
pd.set_option('display.max_rows', None)       # 모든 행 출력
pd.set_option('display.max_columns', None)    # 모든 열 출력
total_info_df

In [ ]:
# 카테고리별 키워드 사전 정의
flavor_categories = {
"간 관련" : ["간","짠맛"],
"불향/자극" : ["숯불", "불향", "불맛", "불", "고추", "강하다","양념"],
"느끼함/지방감" : ["육즙", "기름", "느끼하다","냄새"],
"담백한 맛" : ["깔끔하다", "담백하다"],
"양과 풍부함" : ["적다", "작다", "적당하다", "달다", "가득하다","크기", "두께", "두툼"],
"식감" : ["부드럽다","촉촉하다", "식감", "씹히다"],
"단 맛": ["달다"]
}


def label_flavor_category(index_word):
    for category, keywords in flavor_categories.items():
        for keyword in keywords:
            if keyword in index_word:  # 부분 일치 허용
                return category
    return "기타"

# df.index = df.index.astype(str)
total_info_df["범주"] = total_info_df.index.map(label_flavor_category)
total_info_df

In [ ]:
total_info_df['범주'].unique()

In [ ]:
total_info_df[total_info_df['범주'] == '식감']

In [ ]:
total_info_df.to_csv('./data/맛_키워드_비율_전체.csv')

In [ ]:
df = total_info_df.copy()

In [ ]:
# 5번째 행부터 끝까지 숫자 데이터만 추출해서 합계 계산
numeric_sum = df.iloc[5:, :-1].sum()

# 결과 출력
print(numeric_sum)

맛 표현이 독립적이진 않다보니 다 더했을 때 1인건 아님

ex) 깔끔하고 넉넉해요~~ 의 경우 '깔끔하다'와 '넉넉하다'두 표현이 들어있음

In [ ]:
# 범주 컬럼 기준으로 그룹화한 뒤, 숫자형 컬럼에 대해 합계 계산
grouped_df = df.iloc[5:].groupby('범주').sum()

# 결과 확인
print(grouped_df)

In [ ]:
!pip install matplotlib

In [ ]:
# 1. 나눔글꼴 설치 및 설정


import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np
import os

# ✅ Windows에서 시스템 폰트 직접 설정
font_path = "C:/Windows/Fonts/malgun.ttf"  # 또는 NanumGothic.ttf가 있는 경로
font_name = fm.FontProperties(fname=font_path).get_name()
plt.rcParams['font.family'] = font_name
plt.rcParams['axes.unicode_minus'] = False


# 2. 방사형 차트 기본 설정
categories = grouped_df.index.tolist()
num_vars = len(categories)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

colors = ['#FF8C00', '#DC143C', '#4169E1', '#228B22', '#8A2BE2', '#FF1493']
brands = grouped_df.columns.tolist()

# 3. y축 최대값 계산 (모든 값 중 가장 큰 값)
max_val = grouped_df.values.max()

# 4. Subplot 생성 (3행 2열)
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(12, 14), subplot_kw=dict(polar=True))
axes = axes.flatten()

for idx, brand in enumerate(brands):
    values = grouped_df[brand].tolist()
    values += values[:1]

    ax = axes[idx]
    ax.plot(angles, values, color=colors[idx], linewidth=2)
    ax.fill(angles, values, color=colors[idx], alpha=0.25)

    # 축 설정
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_thetagrids(np.degrees(angles[:-1]), categories)
    ax.set_ylim(0, max_val)  # 모든 그래프에 동일한 y축 범위 적용
    ax.set_title(f"{brand}의 맛 범주 비율", y=1.1)
    ax.grid(True)

# 남는 subplot 제거
for j in range(len(brands), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()
